# Data exploration

EMRIS (Emergency Management & Resource Information System) registers, plans and tracks emergencies in Colombia. This notebook explores the two datasets used to train its models: the UNGRD emergency records and the Colombian Geological Survey seismic catalog. Everything here calls the same cleaning functions the models use.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
DATA = ROOT / "data"

## Load the data

`clean_ungrd` cleans the raw UNGRD file and reports what it discarded. `load_frame` joins the cleaned records with the DIVIPOLA municipality codes and returns the table the models are trained on.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from ml.data import clean_ungrd, load_frame
from ml.features import recorded_impact

cleaned = clean_ungrd(DATA / "ungrd_emergencies_v2.csv")
records = cleaned.records
frame = load_frame(DATA / "ungrd_emergencies_v2.csv", DATA / "divipola_dane.csv")
print(f"Cleaned records: {len(records):,}")
print(f"Discarded: {dict(cleaned.discarded)}")
print(f"Model frame rows: {len(frame):,}")

## Emergencies per month

The number of emergencies registered in each calendar month over the whole period.

In [ ]:
months = pd.to_datetime(records["occurred_on"]).dt.to_period("M")
per_month = months.value_counts().sort_index()
ax = per_month.plot.bar(figsize=(14, 4), width=0.9)
ax.set_xlabel("Month")
ax.set_ylabel("Emergencies")
ticks = list(range(0, len(per_month), 6))
ax.set_xticks(ticks)
ax.set_xticklabels([str(per_month.index[i]) for i in ticks], rotation=45)
plt.show()

## Top 10 event types

The ten most frequent event types across all records.

In [ ]:
top_types = records["event_type"].value_counts().head(10)
ax = top_types.sort_values().plot.barh(figsize=(8, 4))
ax.set_xlabel("Emergencies")
plt.show()

## High-impact share by event type

An emergency is high impact when it has victims, destroyed homes, or people affected or hectares at or above the 90th percentile of its event type. The chart shows the share of high-impact emergencies in each type.

In [ ]:
high_impact = records["is_high_impact"]
print(f"High-impact emergencies: {int(high_impact.sum()):,}")
share_by_type = high_impact.groupby(records["event_type"]).mean().sort_values()
ax = share_by_type.plot.barh(figsize=(8, 5))
ax.set_xlabel("Share of high-impact emergencies")
plt.show()

## Rows with no recorded impact

Some records report none of the nine impact variables. They stay in the data but are left out of the emergency profiles.

In [ ]:
no_impact = int((~recorded_impact(records)).sum())
print(f"Records with no recorded impact: {no_impact:,}")

## Seismic catalog

The seismic catalog lists earthquakes of magnitude 2.5 or more. The first chart counts events per year. The second shows the depth distribution, with a line at the 70 km limit that separates shallow from deep events. Shallow events are the ones that can damage buildings, so only they feed the municipal risk model.

In [ ]:
from ml.seismic import RISK_MIN_MAGNITUDE, SHALLOW_DEPTH_KM, read_catalog

events = list(read_catalog(DATA / "sgc_seismicity_m25.csv"))
print(f"Seismic events: {len(events):,}")

per_year = pd.Series([event.event_time.year for event in events]).value_counts().sort_index()
ax = per_year.plot.bar(figsize=(10, 3.5))
ax.set_xlabel("Year")
ax.set_ylabel("Events")
plt.show()

depths = [event.depth_km for event in events]
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(depths, bins=60)
ax.axvline(SHALLOW_DEPTH_KM, color="red", linestyle="--", label=f"{SHALLOW_DEPTH_KM:.0f} km")
ax.set_xlabel("Depth (km)")
ax.set_ylabel("Events")
ax.legend()
plt.show()

strong = [event for event in events if event.magnitude >= RISK_MIN_MAGNITUDE]
shallow = sum(event.depth_km < SHALLOW_DEPTH_KM for event in strong)
print(f"Events with magnitude >= {RISK_MIN_MAGNITUDE}: shallow {shallow}, deep {len(strong) - shallow}")